<a href="https://colab.research.google.com/github/EvolvingAgentsLabs/agent-forge/blob/main/jit_poc_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===================================================
# CELL 1: Environment Setup
# ===================================================
# @title Step 1: Install Unsloth & Dependencies

# We will use Unsloth's optimized installation for Colab.
# This provides significant speed and memory improvements.
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install other necessary libraries
!pip install -q --no-deps transformers peft accelerate bitsandbytes
!pip install -q requests

print("✅ Dependencies installed with Unsloth.")

# Check the allocated GPU. A T4 or L4 from a free Colab instance is sufficient.
!nvidia-smi

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-rryj4ono/unsloth_de08a37f56c34b4588418da3b8b3bc9a
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-rryj4ono/unsloth_de08a37f56c34b4588418da3b8b3bc9a
  Resolved https://github.com/unslothai/unsloth.git to commit bcb1bcfd6aa9dc7db8910c733cecf0c676b37eb3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.8/184.8 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 17.8 MB/s eta 0:

In [2]:
# ===================================================
# CELL 2: Load the Base Model
# ===================================================
# @title Step 2: Load Qwen2.5-Coder-1.5B with 4-bit Quantization

from unsloth import FastQwen2Model
import torch

# The maximum sequence length the model can handle.
max_seq_length = 8192

# Load the model and tokenizer using Unsloth's highly optimized function.
model, tokenizer = FastQwen2Model.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,      # None for auto detection
    load_in_4bit = True,
)

print("✅ Qwen2.5-Coder-1.5B model loaded successfully with Unsloth.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.8.9: Fast Qwen2 patching. Transformers: 4.55.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


config.json:   0%|          | 0.00/765 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Qwen2.5-Coder-1.5B model loaded successfully with Unsloth.


In [3]:
# ===================================================
# CELL 3: Prepare the Model for Fine-Tuning
# ===================================================
# @title Step 3: Prepare the Model for Fine-Tuning with LoRA

model = FastQwen2Model.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("✅ Model prepared for LoRA fine-tuning.")

Unsloth 2025.8.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ Model prepared for LoRA fine-tuning.


In [4]:
# ===================================================
# CELL 4: The FINAL Fine-Tuning Dataset (for "Prompt List" Generation)
# ===================================================
# @title Step 4: Define the Final "Prompt List" Fine-Tuning Dataset

import json
from datasets import load_dataset, Dataset

# This UNIFIED dataset now teaches the model to generate a JSON list of prompts.
# This is a much simpler and more reliable task for the LORA than generating Python code.
# The runtime will then execute this list of prompts.
raw_dataset = [
    # Example 1: The Core Scenario (Facts, Project Info)
    {
        "messages": [
            {
                "role": "user",
                "content": """You are a memory analysis agent. Analyze the following conversation context and generate a JSON list of prompts that can be used to extract key memories.

Conversation Context:
---
user: Hi, I'm Alex Taylor, user ID `alex123`, and I am the lead on the 'Project Phoenix' initiative.
assistant: Understood. I have noted your name, user ID, and your role on Project Phoenix.
user: Also, please set the deadline for next Friday and assign Sarah Chen as the lead developer.
---"""
            },
            {
                "role": "assistant",
                "content": """[
    {
        "memory_type": "facts",
        "prompt": "Extract key facts from the following text as a single, valid JSON object. Use keys like 'user_name', 'user_id', 'project_name', 'deadline', and 'team_members'.\\n\\nText: {context}"
    },
    {
        "memory_type": "episodes",
        "prompt": "Summarize the key events from the following text as a bulleted list, with each event on a new line.\\n\\nText: {context}"
    }
]"""
            }
        ]
    },
    # Example 2: Technical Preferences and Constraints
    {
        "messages": [
            {
                "role": "user",
                "content": """You are a memory analysis agent. Analyze the following conversation context and generate a JSON list of prompts that can be used to extract key memories.

Conversation Context:
---
user: For Project Nebula, we absolutely must use PostgreSQL for the database. I have 5 years of experience with it.
assistant: Got it. PostgreSQL for Project Nebula.
user: And for the backend, let's stick with Python. I prefer FastAPI.
---"""
            },
            {
                "role": "assistant",
                "content": """[
    {
        "memory_type": "facts",
        "prompt": "Extract the user's technical preferences as a JSON object from the text. Use keys like 'database', 'backend_language', 'backend_framework', and 'user_experience'.\\n\\nText: {context}"
    },
    {
        "memory_type": "episodes",
        "prompt": "Summarize the conversation about technical choices in one sentence.\\n\\nText: {context}"
    }
]"""
            }
        ]
    },
    # Example 3: Ambiguous context requiring interpretation
        {
        "messages": [
            {
                "role": "user",
                "content": """You are a memory analysis agent. Analyze the following conversation context and generate a JSON list of prompts that can be used to extract key memories.

Conversation Context:
---
user: I'm really not happy with the current UI. It feels clunky.
assistant: I understand your frustration. What specific parts are causing issues?
user: The whole checkout process. It needs to be smoother. And make the buttons blue, I like blue.
---"""
            },
            {
                "role": "assistant",
                "content": """[
    {
        "memory_type": "facts",
        "prompt": "Extract the user's explicit preferences as a JSON object from the text. Use keys like 'color_preference' and 'ui_feedback'.\\n\\nText: {context}"
    },
    {
        "memory_type": "episodes",
        "prompt": "Summarize the user's feedback on the UI and checkout process as a single episode.\\n\\nText: {context}"
    }
]"""
            }
        ]
    }
]

dataset = Dataset.from_list(raw_dataset)

# This function formats our raw data into the specific chat template
# that the Qwen model was trained on, which is critical for good performance.
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = True) for convo in convos]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True,)

print(f"✅ Final Prompt List LORA dataset with {len(raw_dataset)} examples created.")

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

✅ Final Prompt List LORA dataset with 3 examples created.


In [5]:
# ===================================================
# CELL 5: Fine-Tune the Memory LORA
# ===================================================
# @title Step 5: Train the Memory-Focused LoRA Adapter

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 150,  # More steps for memory-specific learning
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_memory",
    ),
)

print("Starting Memory LoRA fine-tuning...")
trainer.train()

LORA_ADAPTER_PATH = "adapters/agent_forge_memory"
trainer.save_model(LORA_ADAPTER_PATH)

print(f"\n✅ Memory LoRA adapter fine-tuned and saved to {LORA_ADAPTER_PATH}")

Unsloth: Tokenizing ["text"]:   0%|          | 0/3 [00:00<?, ? examples/s]

Starting Memory LoRA fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 150 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: matias-molinas (matias-molinas-home) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,2.000000
20,0.212800
30,0.008700
40,0.005700
50,0.005200
60,0.005000
70,0.005000
80,0.005000
90,0.004900
100,0.004900



✅ Memory LoRA adapter fine-tuned and saved to adapters/agent_forge_memory


In [6]:
# ===================================================
# CELL 6: The Final `MemoryForgeRuntime` (with Robust Synthesis)
# ===================================================
# @title Step 6: Define the Stateful Memory Forge Runtime (Final Version)

import re
import time
import json
import uuid
from unsloth import FastQwen2Model

class Memory:
    """A standardized class for storing memory items."""
    def __init__(self, content, mem_type, source_id=None):
        self.id = str(uuid.uuid4())
        self.type = mem_type  # 'episode' or 'fact'
        self.content = content
        self.metadata = {
            "created": time.time(),
            "source_id": source_id,
        }
    def __repr__(self):
        return f"Memory(id={self.id}, type='{self.type}', content='{str(self.content)[:50]}...')"

class MemoryForgeRuntime:
    def __init__(self, finetuned_model, tokenizer):
        self.model = finetuned_model
        self.tokenizer = tokenizer
        FastQwen2Model.for_inference(self.model)
        self.memory_store = {}
        print("✅ Stateful Memory Forge Runtime Initialized.")

    def _call_model(self, prompt, **generation_kwargs):
        """Core LLM calling function, handles tokenization, generation, and metrics."""
        print(f"\n>>> Calling Fine-Tuned Model...")
        messages = [{"role": "user", "content": prompt}]

        inputs = self.tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
        ).to("cuda")

        default_kwargs = {
            "max_new_tokens": 1024,
            "use_cache": True,
            "do_sample": True,
            "temperature": 0.1,
        }
        default_kwargs.update(generation_kwargs)

        # Ensure temperature is not passed with greedy decoding
        if not default_kwargs.get("do_sample"):
            default_kwargs.pop("temperature", None)

        start_time = time.time()
        outputs = self.model.generate(**inputs, **default_kwargs)
        content = self.tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        latency = (time.time() - start_time) * 1000

        prompt_tokens = len(inputs["input_ids"][0])
        completion_tokens = len(outputs[0][inputs["input_ids"].shape[-1]:])
        total_tokens = prompt_tokens + completion_tokens

        print(f"<<< Model responded in {latency:.2f}ms. (Total tokens: {total_tokens})")
        return content, latency, total_tokens

    def _update_memory_store(self, extracted_memories, conversation_id):
        """Adds extracted memories to the persistent in-memory store."""
        new_mem_count = 0
        for episode in extracted_memories.get("episodes", []):
            mem = Memory(episode, "episode", source_id=conversation_id)
            self.memory_store[mem.id] = mem
            new_mem_count += 1
        for key, value in extracted_memories.get("facts", {}).items():
            mem = Memory({key: value}, "fact", source_id=conversation_id)
            self.memory_store[mem.id] = mem
            new_mem_count += 1
        print(f"--- Memory store updated with {new_mem_count} new memories. Total memories: {len(self.memory_store)} ---")

    def process_conversation(self, conversation):
        """
        Orchestrates memory extraction using the safe "Prompt-List Executor" pattern.
        """
        print("\n" + "="*20 + " STAGE 1: MEMORY PROMPT GENERATION " + "="*20)
        context_str = "\n".join([f"{msg['role']}: {msg['content']}" for msg in conversation])

        # 1. Generate a JSON list of prompts to execute.
        plan_prompt = f"""You are a memory analysis agent. Analyze the following conversation context and generate a JSON list of prompts that can be used to extract key memories.

Conversation Context:
---
{context_str}
---"""

        prompt_list_str, _, _ = self._call_model(plan_prompt, do_sample=False)

        try:
            # Clean the output in case the model adds markdown formatting
            clean_prompt_list_str = re.sub(r"```json\n|```", "", prompt_list_str).strip()
            prompt_list = json.loads(clean_prompt_list_str)
            print(f"--- Prompt List Generated ---\n{json.dumps(prompt_list, indent=2)}\n--------------------------")
        except json.JSONDecodeError:
            print(f"!!! CRITICAL FAILURE: Model failed to generate a valid JSON prompt list. Raw output:\n{prompt_list_str}")
            return {"error": "Prompt list generation failed."}

        # 2. Execute the list of prompts safely.
        print("\n" + "="*20 + " STAGE 2: SAFE PROMPT EXECUTION " + "="*20)
        extracted_memories = {"episodes": [], "facts": {}}
        for task in prompt_list:
            prompt = task["prompt"].format(context=context_str)
            memory_type = task["memory_type"]

            # Execute the prompt
            print(f"--- Executing prompt for '{memory_type}' memories ---")
            result, _, _ = self._call_model(prompt, do_sample=(memory_type != "facts"))

            # Store the result
            if memory_type == "episodes":
                extracted_memories["episodes"].extend([ep.strip('- ').strip() for ep in result.split('\n') if ep.strip()])
            elif memory_type == "facts":
                try:
                    clean_result = re.sub(r"```json\n|```", "", result).strip()
                    extracted_memories["facts"].update(json.loads(clean_result))
                except json.JSONDecodeError:
                    print(f"--- Warning: Could not parse facts JSON: {result} ---")
                    extracted_memories["facts"]["raw_unparsed_facts"] = result

        print(f"--- Structured Memory Extracted ---\n{json.dumps(extracted_memories, indent=2)}\n---------------------------------")

        self._update_memory_store(extracted_memories, str(uuid.uuid4()))
        return extracted_memories

    def recall_and_answer(self, user_prompt):
        """Finds relevant memories and uses them to answer a new prompt."""
        print("\n" + "="*20 + " STAGE 3: RECALL & SYNTHESIS " + "="*20)

        # Simple keyword-based recall for the POC
        query_words = set(user_prompt.lower().split())
        recalled = []
        for mem in self.memory_store.values():
            if any(word in str(mem.content).lower() for word in query_words):
                recalled.append(mem.content)
        print(f"Found {len(recalled)} relevant memories.")

        # THE FINAL FIX: A more robust "few-shot" prompt to guide the final synthesis.
        # This provides a clear example of the desired output format, preventing the
        # code-specialized model from defaulting to JSON.
        final_prompt = f"""You are a response synthesizer. Your only job is to answer the user's question using ONLY the facts from the 'Relevant Memories' section.
Your response MUST be a single, conversational sentence. DO NOT output JSON, lists, or code.
If the memories do not contain the answer, respond with "I do not have that information in my memory."

---
EXAMPLE INPUT:
User's Question: What is my user ID?
Relevant Memories:
[
  {{
    "user_id": "alex123"
  }}
]
EXAMPLE OUTPUT:
Your user ID is alex123.
---

Here is the real task:

User's Question: {user_prompt}

Relevant Memories:
{json.dumps(recalled, indent=2) if recalled else "No relevant memories found."}

Final Answer:"""

        final_answer, _, _ = self._call_model(final_prompt)

        print("\n" + "="*20 + " FINAL ANSWER " + "="*20)
        print(final_answer)
        return final_answer

In [7]:
# @title Step 7: Demonstrate the Memory System in a Multi-Turn Scenario

def demonstrate_memory_system():
    print("="*60 + "\n        STATEFUL AGENT (CONTEXT ANALYZER) DEMO\n" + "="*60)

    runtime = MemoryForgeRuntime(
        finetuned_model = model,
        tokenizer = tokenizer
    )

    # --- CONVERSATION 1: Establish Facts & Preferences ---
    print("\n\n" + "#"*20 + " CONVERSATION 1: LEARNING " + "#"*20)
    conversation_1 = [
        {"role": "user", "content": "Hi! I'm Alex Taylor, my user ID is alex123. I'm the lead on Project Phoenix."},
        {"role": "assistant", "content": "Great to meet you Alex! I'll remember that."},
        {"role": "user", "content": "Set the deadline for next Friday, and assign Sarah Chen as the frontend lead."}
    ]
    runtime.process_conversation(conversation_1)

    # --- CONVERSATION 2: Test Recall & Synthesis ---
    print("\n\n" + "#"*20 + " CONVERSATION 2: RECALLING & REASONING " + "#"*20)
    user_question = "What's the deadline for my project and who is the lead developer?"
    final_answer = runtime.recall_and_answer(user_question)

    print("\n" + "="*60 + "\n                  DEMONSTRATION SUMMARY\n" + "="*60)
    print(f"User Question: {user_question}")
    print("-" * 60)
    print(f"Final Agent Answer: {final_answer}")
    print("="*60)

# --- Run the main function ---
demonstrate_memory_system()

        STATEFUL AGENT (CONTEXT ANALYZER) DEMO
✅ Stateful Memory Forge Runtime Initialized.


#################### CONVERSATION 1: LEARNING ####################

==================== STAGE 1: MEMORY PROMPT GENERATION ====================

>>> Calling Fine-Tuned Model...
<<< Model responded in 9709.70ms. (Total tokens: 235)
--- Prompt List Generated ---
[
  {
    "memory_type": "facts",
    "prompt": "Extract key facts from the following text as a single, valid JSON object. Use keys like 'user_name', 'user_id', 'project_name', 'deadline', and 'team_members'.\n\nText: {context}"
  },
  {
    "memory_type": "episodes",
    "prompt": "Summarize the key events from the following text as a bulleted list, with each event on a new line.\n\nText: {context}"
  }
]
--------------------------

==================== STAGE 2: SAFE PROMPT EXECUTION ====================
--- Executing prompt for 'facts' memories ---

>>> Calling Fine-Tuned Model...
<<< Model responded in 3324.26ms. (Total tokens: 175)
-